In [ ]:
#Libraries installation
!pip install -U scikit-learn
!pip install librosa matplotlib umap-learn

Uploading dataset



In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import pandas as pd

file_path = '/content/drive/My Drive/cse425/dataset_final.csv'
data = pd.read_csv(file_path)

mfcc_columns = [f'mfcc_{i}' for i in range(1, 14)]
mfcc_features = data[mfcc_columns]
print(mfcc_features.head())

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import umap
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

Correlation Heatmap

In [ ]:
mfcc_columns = [f'mfcc_{i}' for i in range(1, 14)]
mfcc_features = data[mfcc_columns]

correlation_matrix = mfcc_features.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of MFCC Features')
plt.show()

In [ ]:
scaler = StandardScaler()
mfcc_scaled = scaler.fit_transform(mfcc_features)

latent_dim = 2

class VAE(models.Model):
    def __init__(self, latent_dim, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.latent_dim = latent_dim

        self.encoder_dense_1 = layers.Dense(128, activation='relu')
        self.encoder_dense_2 = layers.Dense(64, activation='relu')
        self.z_mean = layers.Dense(latent_dim, name='z_mean')
        self.z_log_var = layers.Dense(latent_dim, name='z_log_var')

        self.decoder_dense_1 = layers.Dense(64, activation='relu')
        self.decoder_out = layers.Dense(mfcc_scaled.shape[1], activation='sigmoid')

    def call(self, inputs):
        x = self.encoder_dense_1(inputs)
        x = self.encoder_dense_2(x)
        z_mean = self.z_mean(x)
        z_log_var = self.z_log_var(x)

        z = self.sampling([z_mean, z_log_var])

        h_decoded = self.decoder_dense_1(z)
        x_decoded_mean = self.decoder_out(h_decoded)

        self.add_loss(self.vae_loss(inputs, x_decoded_mean, z_mean, z_log_var))

        return x_decoded_mean

    def sampling(self, args):
        z_mean, z_log_var = args
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

    def vae_loss(self, inputs, x_decoded_mean, z_mean, z_log_var):
        xent_loss = mfcc_scaled.shape[1] * tf.keras.losses.binary_crossentropy(inputs, x_decoded_mean)

        kl_loss = - 0.5 * tf.reduce_mean(z_log_var - tf.square(z_mean) - tf.exp(z_log_var) + 1)

        return xent_loss + kl_loss

vae_model = VAE(latent_dim)
vae_model.compile(optimizer='rmsprop')
vae_model.fit(mfcc_scaled, epochs=50, batch_size=64)

Latent_dim 1 to 8


In [ ]:
input_shape = (13,)
inputs = layers.Input(shape=input_shape)

x = layers.Dense(64, activation='relu')(inputs)
x = layers.Dense(32, activation='relu')(x)
z = layers.Dense(8, name='z')(x)

encoder = models.Model(inputs, z)

latent_features = encoder.predict(mfcc_scaled)

latent_features_df = pd.DataFrame(latent_features, columns=['latent_dim_1', 'latent_dim_2' , 'latent_dim_3', 'latent_dim_4', 'latent_dim_5', 'latent_dim_6', 'latent_dim_7', 'latent_dim_8'])
print(latent_features_df.head())

Silhouette Score


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans_labels = kmeans.fit_predict(latent_features)

sil_score = silhouette_score(latent_features, kmeans_labels)
print(f'Silhouette Score: {sil_score}')

TSNE PLOTTING


In [ ]:
tsne = TSNE(n_components=2, random_state=42)
latent_2d = tsne.fit_transform(latent_features)

plt.figure(figsize=(8, 6))
plt.scatter(latent_2d[:, 0], latent_2d[:, 1], c=kmeans_labels, cmap='viridis', marker='o')
plt.title('t-SNE Clustering of Latent Features')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.show()

UMAP PLOTTING


In [ ]:
umap_model = umap.UMAP(n_components=2, random_state=42)
latent_2d_umap = umap_model.fit_transform(latent_features)

plt.figure(figsize=(8, 6))
plt.scatter(latent_2d_umap[:, 0], latent_2d_umap[:, 1], c=kmeans_labels, cmap='viridis', marker='o')
plt.title('UMAP Clustering of Latent Features')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.show()

PCA + K-MEANS Silhouette Score


In [ ]:
pca = PCA(n_components=2, random_state=42)
latent_2d_pca = pca.fit_transform(latent_features)

kmeans_pca = KMeans(n_clusters=3, random_state=42)
kmeans_labels_pca = kmeans_pca.fit_predict(latent_2d_pca)


sil_score_pca = silhouette_score(latent_2d_pca, kmeans_labels_pca)
print(f'Silhouette Score (PCA + K-Means): {sil_score_pca}')

PCA+ K-MEANS Clustering

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(latent_2d_pca[:, 0], latent_2d_pca[:, 1], c=kmeans_labels_pca, cmap='viridis', marker='o')
plt.title('PCA + K-Means Clustering of Latent Features')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.show()

Calinski-Harabasz Index (PCA + K-Means)

In [ ]:
ch_score_pca = calinski_harabasz_score(latent_2d_pca, kmeans_labels_pca)
print(f'Calinski-Harabasz Index (PCA + K-Means): {ch_score_pca}')